In [1]:
import importlib as il
import numpy as np
import more_itertools as mit

import gurobipy as gp

import gurobi_utils as gu
import dikin_utils as du
import plot_utils as pu

import example_loader as el
import miplib_loader as ml
import jsplib_loader as jl

%matplotlib inline
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

<gurobipy.Env, Parameter changes: WLSAccessID=(user-defined), WLSSecret=(user-defined), LicenseID=2586148, OutputFlag=0>

What we want to do here:
1. Find an LP optimum for some problem.
2. Back away from that optimum into the interior sqrt(n)/2 distance.
3. Find the eigenvectors for Dikin's H at that point.
4. Round those to be integer values.
5. Run the LLL reduction on that result.
6. Make that be unimodular if it's not.
7. Transform the original problem and run the MIP solver for it.

In [2]:
def retreat_from_optimum_via_average_vector(V: np.ndarray, x: np.ndarray, target_distance):
    """Retreat from the optimum by moving in the direction of the average vector."""
    # this isn't perfect at the moment in that we should normalize the columns of V first.
    # that way each vector would contribute equally to the average.
    # however, some columns are zero-length, and we would have to handle that case.
    
    # normalize the columns first so that they all contribute equally:
    V /= np.linalg.norm(V, axis=0)

    avg = np.mean(V, axis=1)
    nrm = np.linalg.norm(avg) + 1e-5
    return x - avg * target_distance / nrm

def find_corner(relaxed: gp.Model, int_vars, int_var_idx):
    basis = gu.read_basis(relaxed)
    tableau, col_to_var_idx, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=0)
    negated_vars = [basis[nr] for nr in negated_rows]

    # the current understanding (from nlhdlr_quadratic.c in SCIP): 
    # negate all columns with variables at status -1
    # and negate all columns match slack variables of type <
    variables = relaxed.getVars()
    constraints = relaxed.getConstrs()
    for col, j in enumerate(col_to_var_idx):
        if j < len(variables):
            # print("Var INFO:", variables[j].VarName, "VBasis", variables[j].VBasis, "LB", variables[j].LB, "UB", variables[j].UB)
            if variables[j].VBasis == -2:
                # tableau[:, col] = variables[j].UB - tableau[:, col]
                tableau[:, col] = -tableau[:, col]
                print("   VBasis at -2 for", variables[j].VarName, variables[j].UB)
    #         if variables[j].VBasis == -1:  # not sure what to do with VBasis=-3
    #             tableau[:, col] = -tableau[:, col]  # variables[j].LB
    #             if variables[j].LB != 0.0:
    #                 print("Warning: LB is nonzero for variable", variables[j].VarName, "LB", variables[j].LB, "UB", variables[j].UB)
        else:
            constraint = constraints[j - len(variables)]
    #         # this might not be right: scip has status and tests for A_i*x being at lower or upper bound
    #         # if np.isclose(constraint.Slack, 0.0, atol=tol):
    #         #     tableau[:, col] = -tableau[:, col]
            if constraint.Sense == '>':  # Achterberg said lt and lte are standard; should just need to flip gt
                tableau[:, col] = -tableau[:, col]
                print("   GTE Constraint found for", constraint.ConstrName)

    # drop the rows of non-integer variables:
    to_drop = [i for i, b in enumerate(basis) if b not in int_var_idx]
    tableau = np.delete(tableau, to_drop, axis=0)  # TODO: don't even bother to read them in
    basis = np.delete(basis, to_drop) # update basis to match tableau

    # we want all the integer variables in order, assuming x, y as the first two.
    # however, some integer variables may be columns in the tableau, which is problematic.

    basis, tableau = mit.sort_together([basis, tableau], key_list=[0]) #, key=int_var_idx.get)
    tableau = np.array(tableau)

    sv = [int_vars[int_var_idx[b]].X for b in basis]
    return tableau, np.array(sv)

def make_primal_dual_values(mdl: gp.Model):
    """Extract primal and dual values from the model."""
    primal = np.array([v.X for v in mdl.getVars()])
    dual_cons = np.array([c.Pi for c in mdl.getConstrs()])
    # dual_vars = np.array([v.RC for v in mdl.getVars()])
    return primal, dual_cons

def get_A_b_c_l_u(mdl: gp.Model):
    mdl.update()
    A = mdl.getA()
    b = np.array(mdl.getAttr("RHS")).reshape(-1, 1)
    c = np.array(mdl.getAttr("Obj"))
    l = np.array(mdl.getAttr("LB"))
    u = np.array(mdl.getAttr("UB"))
    return A, b, c, l , u



In [ ]:
il.reload(el)
il.reload(gu)
il.reload(pu)
il.reload(du)

def shrink(mdl: gp.Model, diagonal_distance, percent_of_diagonal):
    assert diagonal_distance > 0.0 and percent_of_diagonal > 0.0 and percent_of_diagonal < 0.5
    mdl.update()
    result = mdl.copy()
    if result.NumIntVars > 0:
        _, _ = gu.relax_int_or_bin_to_continuous(result)
    result.update()
    distance = diagonal_distance * percent_of_diagonal
    for v in result.getVars():
        if v.UB - v.LB < distance * 2.0:
            gap = (v.UB - v.LB) * percent_of_diagonal
            v.LB += gap
            v.UB -= gap
        else:
            if v.LB > -gp.GRB.INFINITY:
                v.LB += distance
            if v.UB < gp.GRB.INFINITY:
                v.UB -= distance
    for c in result.getConstrs():
        lhs = result.getRow(c)
        coeffs = np.array([lhs.getCoeff(i) for i in range(lhs.size())])
        if c.Sense == '<':
            c.RHS -= distance * np.linalg.norm(coeffs)
        elif c.Sense == '>':
            c.RHS += distance * np.linalg.norm(coeffs)
    result.update()
    return result

instances = el.get_instances(env)
for instance in list(instances.values()):
    model = instance.as_gurobi_model()
    model.update()
    niv = model.NumIntVars
    _, _ = gu.relax_int_or_bin_to_continuous(model)
    model.update()
    # model.optimize()
    # x = np.array([v.X for v in model.getVars()])
    print("Running:", model.ModelName)
    copy = shrink(model, np.sqrt(niv), 0.3)
    copy.optimize()
    x2 = np.array([v.X for v in copy.getVars()])
    # V, x = find_corner(model, int_vars, int_var_idx)
    # x2 = retreat_from_optimum_via_average_vector(V, x, np.sqrt(int_vars.size) * 0.4)
    # x, y = make_primal_dual_values(model)
    gu.standardize_gt_to_lt(model)
    A, b, c, lb, ub = get_A_b_c_l_u(model)
    # x2, its = du.least_squares_interior(A, b, x, l, u, d=np.sqrt(int_vars.size) * 0.4, infinity=gp.GRB.INFINITY)
    # x2, its = du.reverse_interior_point_gpt2(A, b, c, l, u, x, y, target_distance=np.sqrt(int_vars.size) * 0.4,
    #                                          infinity=10, is_maximizing=model.ModelSense == gp.GRB.MAXIMIZE)
    print("   Retreat to:", x2)
    fig = pu.plot_constraints_lte(model.ModelName, A, b, lb, ub, points=[x2[:2]])
    du.plot_ellipse(A, b, lb, ub, x2, fig=fig)

    # gu.standardize_gt_to_lt(copy)
    # A, b, c, l, u = get_A_b_c_l_u(copy)
    # fig = pu.plot_constraints_lte(model.ModelName, A, b, l, u, points=[x2[:2]], fig=fig)


    # could assert x2 is feasible




In [4]:
il.reload(du)
# test = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
# test = np.array([[.1, -1, .3], [1, 0, 5], [1, 2, 6]])
test = np.array([[1, -1, 3], [1, 0, 5], [1, 2, 6]])
# test = np.array([[1, 0, 5], [1, -1, 3], [1, 2, 6]])

# U = du.CLLL(test)
import scipy.sparse as sps
test = sps.csc_matrix(test)
U = du.CLLL(test)
test.toarray(), U, np.linalg.det(U)

/home/brannon/Documents/Research/venv10/lib/python3.10/site-packages/scipy/sparse/_index.py:188: SparseEfficiencyWarning: Changing the sparsity structure of a csc_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray_sparse(i, j, x)


(array([[ 8,  9,  3],
        [ 9, 10,  5],
        [ 8,  9,  6]]),
 array([[ 4,  5,  0],
        [-1, -1,  0],
        [ 1,  1,  1]], dtype=int32),
 1.0)

$$
H(x_0) = A^\top W(x_0) A + D(x_0),
$$
where:

1. $( W(x_0) )$ is a diagonal weight matrix for the inequality constraints $( Ax \leq b )$, defined as:
$$
W(x_0) = \text{diag}\left(\frac{1}{(b_i - A_i^\top x_0)^2}\right), \quad i = 1, 2, \dots, m.
$$
Here, $( b_i - A_i^\top x_0 > 0 )$ because $( x_0 )$ is an interior point.

2. $( D(x_0) )$ is a diagonal matrix that accounts for the bounds $( l \leq x \leq u )$, defined as:
$$
D(x_0) = \text{diag}\left( \frac{1}{(x_{0j} - l_j)^2} + \frac{1}{(u_j - x_{0j})^2} \right), \quad j = 1, 2, \dots, n.
$$
The terms $( x_{0j} - l_j > 0 )$ and $( u_j - x_{0j} > 0 )$ hold because $( x_0 )$ is an interior point.

For the transformation, define $x=Uy + x_0$. Then:

The variable bounds transformation: $U^{-1} (\ell_x - x_0) \leq y \leq U^{-1} (u_x - x_0)$

Constraints: $A U y \leq b - A x_0$

Objective: $(c^T U) y + c^T x_0 + c_0 = (U^T c)^T y + c^T x_0 + c_0$

In [ ]:
# make it be round:

il.reload(el)
il.reload(gu)
il.reload(pu)
il.reload(du)

def apply_transform(old_model: gp.Model, U: np.ndarray, x0):
    """Apply the transformation U to the model."""
    old_model.update()
    # A, b, c, l, u = get_A_b_c_l_u(result)

     # Get original data
    A = old_model.getA()  # Constraint coefficient matrix (as a scipy.sparse matrix)
    rhs = old_model.getAttr("RHS")  # Right-hand side vector
    sense = old_model.getAttr("Sense")  # Constraint senses (<=, >=, =)
    lb = np.array(old_model.getAttr("LB"))
    ub = np.array(old_model.getAttr("UB"))

    vtypes = []
    for idx, v in enumerate(old_model.getVars()):
        if v.VType == gp.GRB.BINARY:
            lb[idx] = 0.0
            ub[idx] = 1.0
            vtypes.append(gp.GRB.INTEGER)
        else:
            vtypes.append(v.VType)
    vtypes = np.array(vtypes)

    lb -= x0
    ub -= x0
    lb[lb < -gp.GRB.INFINITY] = -gp.GRB.INFINITY  # we can't have infinities when we multiply by 0
    ub[ub > gp.GRB.INFINITY] = gp.GRB.INFINITY

    # Ensure unimodular_matrix has the correct dimensions
    num_vars = old_model.NumVars
    if U.shape != (num_vars, num_vars):
        raise ValueError("Unimodular matrix must have dimensions matching the number of variables.")

    # Create a new model
    new_model = gp.Model(name=f"{old_model.ModelName}_transformed")
    U_inv = np.linalg.inv(U)
    lb = U_inv @ lb
    ub = U_inv @ ub

    # Add new variables y corresponding to the transformed space
    y_vars = new_model.addMVar(num_vars, lb=lb, ub=ub, vtype=vtypes, name=f"y")

    # Compute AU
    A_transformed = A @ U
    b_deduction = A @ x0

    # Add the transformed constraints AUy <= b (or other senses)
    for i in range(A_transformed.shape[0]):
        expr = A_transformed[i, :] @ y_vars
        if sense[i] == '<':
            new_model.addConstr(expr <= rhs[i] - b_deduction[i])
        elif sense[i] == '>':
            new_model.addConstr(expr >= rhs[i] - b_deduction[i])
        elif sense[i] == '=':
            new_model.addConstr(expr == rhs[i] - b_deduction[i])

    # Transform the objective
    obj_coeffs = np.array([v.Obj for v in model.getVars()])  # more efficient way?
    transformed_obj = obj_coeffs @ U  # Transform the objective coefficients
    obj = model.getObjective()
    new_model.setObjective(transformed_obj @ y_vars + obj_coeffs @ x0 + obj.getConstant(), old_model.ModelSense)

    return new_model

instances = el.get_instances(env)
# instances = jl.get_instances()
for instance in list(instances.values())[0:3]:
    # model = instance.as_gurobi_balas_model(use_big_m=True, env=env)
    model = instance.as_gurobi_model()
    print("Running:", model.ModelName)

    copy = shrink(model, np.sqrt(niv), 0.3)
    copy.optimize()
    x2 = np.array([v.X for v in copy.getVars()])

    gu.standardize_gt_to_lt(model)
    A, b, c, l, u = get_A_b_c_l_u(model)
    
    H = du.compute_H(A, b, l, u, x2)
    import scipy.linalg as spl
    H2 = spl.sqrtm(H.toarray())  # TODO: add a more efficient eigenvalue decomposition for the sqrt
    H2 = np.round(H2 * 1000)
    U = du.CLLL(H2)  # modifies H2
    print("Determinant:", np.linalg.det(U))

    model2 = apply_transform(model, U, np.zeros_like(x2))
    A, b, c, l, u = get_A_b_c_l_u(model2)

    fig = pu.plot_constraints_lte(model.ModelName, A, b, l, u, points=[x2[:2]])
    du.plot_ellipse(A, b, l, u, x2, fig=fig)


Define $x=Uy + x_0$. Then:

The variable bounds transformation: $U^{-1} (\ell_x - x_0) \leq y \leq U^{-1} (u_x - x_0)$

Constraints: $A U y \leq b - A x_0$

Objective: $(c^T U) y + c^T x_0 + c_0$

In [18]:
import jsplib_loader as jl
instances = jl.get_instances()
instance = instances["abz5"]
model = instance.as_gurobi_model()
print("Running:", model.ModelName)

copy = shrink(model, np.sqrt(niv), 0.3)
copy.optimize()
x2 = np.array([v.X for v in copy.getVars()])

gu.standardize_gt_to_lt(model)
A, b, c, l, u = get_A_b_c_l_u(model)

H = du.compute_H(A, b, l, u, x2)
import scipy.linalg as spl
H2 = spl.sqrtm(H.toarray())  # TODO: add a more efficient eigenvalue decomposition for the sqrt
H2 = np.round(H2 * 1000)
U = du.CLLL(H2)  # modifies H2
print("Determinant:", np.linalg.det(U))

model2 = apply_transform(model, U, x2)
model2.optimize()

Set parameter AggFill to value 10
Set parameter GomoryPasses to value 1
Running: abz5
   Relaxed 1000 variables on abz5_copy
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "EndeavourOS")

CPU model: Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
GomoryPasses  1
AggFill  10

Academic license 2586148 - for non-commercial use only - registered to br___@vt.edu
Optimize a model with 1000 rows, 1101 columns and 2900 nonzeros
Model fingerprint: 0xfe912e72
Coefficient statistics:
  Matrix range     [1e+00, 8e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [3e-01, 7e-01]
  RHS range        [5e+01, 5e+03]
Presolve removed 451 rows and 550 columns
Presolve time: 0.01s
Presolved: 549 rows, 1001 columns, 2007 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.6285695e+02   1.190179e+03   0.000000e+00      0s
   

AttributeError: 'gurobipy.Var' object has no attribute 'Index'